# Figure panels for tracks data processing and effective $k_\mathrm{obs}$ fits

## Imports

- Calcium tracks data processing functions are in the `analysis/tracks.jl` module. 
- Functions that make final, detailed plots are in `analysis/tracksPlots.jl`. 
- Functions to fit a one-step model for effective activation rates $k_\mathrm{obs}$ are in `analysis/modelsML.jl`. 

In [ ]:
using Plots: Plots as pl
using Printf: @sprintf
using Measures: mm

# Data processing
module tracksAnalysis
    include("analysis/tracks.jl")
end
using .tracksAnalysis: preptrajdata, findactivtimes_thresh, 
    cleannames, get_csv_file_names,  prepalldsets, concfromstr,
    prepare_all_cdfs

# import small utilities from the plotting scripts
module tracksPlots
    include("analysis/tracksPlots.jl")
end
using .tracksPlots: plot_trajs_activ_cdf, plot_kobs_fits, plot_kobs_vs_concentration


# Model definitions and log-likelihood
module kObsML
    include("analysis/kObsML.jl")
end
using .kObsML: expocdf, fitkobs


In [ ]:
# Default styles
pl.default(
    labelfontsize = 14,
    tickfontsize = 12,
    framestyle = :axes,  # https://docs.juliaplots.org/stable/gallery/gr/generated/gr-ref034/
    yminorgrid = false,
    xminorgrid = false,
    xgrid = false,
    ygrid = false,
    size = (550, 450),
    linewidth = 2.0,
    legendfontsize=12, 
    fontfamily="Arial"
)

# Default analysis parameters
default_fluo_factor = 0.8
cdfthreshold = 0.75
# Filtering parameters: maximal start time, minimal traj length
minlength = 50
maxstarttime = 30.0
filter_params = [minlength, maxstarttime]

# Example trajectories and thresholding
All trajectories in light gray on one plot (all with annotated transition time perhaps), one colored with thresholding to emphasize the procedure. 

In [ ]:
# Main fitting and plotting function for one example
function show_tracks_thresholding(
        filepath, plotfolder; 
        highlight=1, 
        fluothresh=default_fluo_factor, 
        filterpms=filter_params, 
        cdfthresh=cdfthreshold, 
        dosave=false
    )
    # highlight: which track to highlight. 
    greentrajs = preptrajdata(filepath; ch="MEAN_INTENSITY_CH2", filterpms=filterpms)  # Calcium fluorescence
    fi = basename(filepath)

    activtimes, activfluos, thresh = findactivtimes_thresh(
        greentrajs; fracabove=fluothresh)
    backgnd = thresh / (1.0 + fluothresh)

    # Make plot
    shortnames, lblconcs = cleannames([fi])
    fi2 = replace(shortnames[1], "." => "-")
    fi2 = replace(fi2, " " => "_")
    ppp = plot_trajs_activ_cdf(greentrajs, activtimes, activfluos, thresh, backgnd, 
                    highlight; conclbl=shortnames[1])
    
    display(ppp)
    if dosave
        pl.savefig(ppp, joinpath(plotfolder, "tracks_example_thresholding_$(fi2).pdf"))
        println("Saved tracks examples")
    end
end

In [ ]:
datadir = joinpath("data", "JR064_tracks")
plotfolder = joinpath("figures", "final_tracks_panels")

# Get a cleaned up, sorted list of data file names and paths
fnames, fpaths = get_csv_file_names(datadir)
shortnames, lblconcs = cleannames(fnames)

dset_id = 3
chosenfile = fpaths[dset_id]
println("File analyzed: ", fnames[dset_id])

show_tracks_thresholding(
    chosenfile, plotfolder;
    highlight=12, 
    dosave=true, 
)

# Effective activation rate $k_\mathrm{obs}$ fitting


## Main $k_\mathrm{obs}$ analysis function

In [ ]:
### MAIN ANALYSIS FUNCTION ###

function fit_ml_all_datasets(; 
    datadir="data/JR064_tracks",
    plotdir="figures/final_tracks_panels/",
    fracabove=default_fluo_factor,
    cdfthresh=cdfthreshold)
    
    fnames, fpaths = get_csv_file_names(datadir)

    println("\n" * "="^70)
    println("MAXIMUM LIKELIHOOD FITTING - JR064 DATASETS")
    println("="^70)
    println("\nLoading and preprocessing datasets...")
    
    # Step 1: Preprocess all datasets
    tactiv_alldsets, durations, nicenames = prepalldsets(fpaths; 
        fracabove=fracabove, disp=false)
    n_dsets = length(tactiv_alldsets)
    println("\nLoaded $n_dsets datasets:")
    for (i, name) in enumerate(nicenames)
        println("  [$i] $name")
    end

    # Step 2: Fit the exponential model to each dataset separately
    println("\n" * "-"^70)
    println("Fitting exponential model for each dataset separately...")
    println("-"^70)
    fitres = fitkobs(tactiv_alldsets, nicenames, durations, cdfthresh)
    fit_params, fit_errors, prep_cdf_res = fitres

    # Select one dataset per concentration for plotting
    selected_indices = Int[7, 4, 3, 1]
    sind = join((string(a) for a in selected_indices), "")
    selected_tactiv = tactiv_alldsets[selected_indices]
    selected_durations = durations[selected_indices]
    selected_names = nicenames[selected_indices]
    selected_fit_params = fit_params[:, selected_indices]
    selected_fit_errors = fit_errors[:, selected_indices]

    println("\nSelected $(length(selected_indices)) datasets for plotting:")
    for (j, name) in enumerate(selected_names)
        println("  [$j] $name")
    end

    # Step 3: Create comparison plots
    println("\n" * "-"^70)
    println("Generating plots...")
    println("-"^70)

    selected_cdf_prep = collect(a[selected_indices] for a in prep_cdf_res)

    p_all = plot_kobs_fits(selected_cdf_prep, selected_names, selected_fit_params, 
                        expocdf, selected_durations)
    plot_file = joinpath(plotdir, "exp_cdf_selected_datasets_$(sind).pdf")
    pl.savefig(p_all, plot_file)
    display(p_all)
    println("  Saved: $plot_file")

    p_conc = plot_kobs_vs_concentration(selected_names, 
                selected_fit_params, selected_fit_errors, prep_cdf_res[2])
    plot_file = joinpath(plotdir, "exp_k_vs_concentration_$(sind).pdf")
    pl.savefig(p_conc, plot_file)
    display(p_conc)
    println("  Saved: $plot_file")

    println("\n" * "="^70)
    println("Analysis complete!")
    println("="^70 * "\n")

    return (fit_params, nicenames, tactiv_alldsets)
end



In [ ]:
fit_ml_all_datasets();